# Exercise 4 — Distributed K-Means Clustering: Performance Analysis

This notebook analyzes serial vs MPI-distributed K-Means on the **Covertype dataset** (581,012 samples × 54 features, standardized).

- **Serial**: full dataset on one process — assign → update → convergence check  
- **Distributed (MPI, 4 processes)**: data partitioned by rows; local assign + `MPI.Allreduce` for centroid update

K values tested: **3, 5, 7** | Max iterations: **50** | Convergence: centroid shift < tolerance

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
COLORS = plt.cm.tab10.colors

## 1. Load Benchmark Results

In [ ]:
df = pd.read_csv('benchmark_results.csv')
df

In [ ]:
serial_df = df[df['strategy'] == 'serial'].copy()
dist_df   = df[df['strategy'] == 'distributed'].copy()
k_values  = sorted(df['k'].unique())
n_workers = int(dist_df['num_workers'].iloc[0]) if len(dist_df) > 0 else 4
print(f'K values tested: {k_values}')
print(f'MPI processes:   {n_workers}')

## 2. Execution Time, Speedup, and Inertia

In [ ]:
def get_vals(source_df, col):
    return [source_df[source_df['k'] == k][col].values[0]
            if len(source_df[source_df['k'] == k]) > 0 else np.nan
            for k in k_values]

ser_times    = get_vals(serial_df, 'time_seconds')
dist_times   = get_vals(dist_df,   'time_seconds')
speedups     = get_vals(dist_df,   'speedup')
ser_inertia  = get_vals(serial_df, 'inertia')
dist_inertia = get_vals(dist_df,   'inertia')

x     = np.arange(len(k_values))
width = 0.35

fig, axes = plt.subplots(1, 3, figsize=(17, 5))

# Execution time grouped by k
ax = axes[0]
bars1 = ax.bar(x - width / 2, ser_times,  width, label='Serial',
               color=COLORS[0], edgecolor='black')
bars2 = ax.bar(x + width / 2, dist_times, width,
               label=f'Distributed ({n_workers} proc)', color=COLORS[1], edgecolor='black')
for bar, val in zip(list(bars1) + list(bars2), ser_times + dist_times):
    if not np.isnan(val):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.02,
                f'{val:.1f}s', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_xticks(x)
ax.set_xticklabels([f'k={k}' for k in k_values])
ax.set_ylabel('Time (s)')
ax.set_title('Execution Time per K Value')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)

# Speedup vs k
ax = axes[1]
ax.plot(k_values, speedups, marker='o', linewidth=2.5, color=COLORS[2],
        markersize=9, label='Achieved')
ax.axhline(y=n_workers, color='black', linestyle='--', linewidth=1.5,
           label=f'Ideal ({n_workers}\u00d7)')
ax.axhline(y=1.0, color='gray', linestyle=':', linewidth=1.2, label='No speedup')
for ki, su in zip(k_values, speedups):
    if not np.isnan(su):
        ax.annotate(f'{su:.2f}\u00d7', (ki, su),
                    textcoords='offset points', xytext=(0, 10), ha='center', fontsize=10)
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Speedup (\u00d7)')
ax.set_title('Speedup vs K')
ax.set_xticks(k_values)
ax.set_ylim(0, n_workers * 1.3)
ax.legend()
ax.grid(True, alpha=0.3)

# Inertia vs k
ax = axes[2]
ax.plot(k_values, ser_inertia,  marker='o', linewidth=2, color=COLORS[0], label='Serial')
ax.plot(k_values, dist_inertia, marker='s', linewidth=2, color=COLORS[1],
        linestyle='--', label='Distributed')
ax.set_xlabel('Number of Clusters (k)')
ax.set_ylabel('Inertia (within-cluster SS)')
ax.set_title('Inertia vs K (lower = tighter clusters)')
ax.set_xticks(k_values)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('performance_kmeans.png', bbox_inches='tight')
plt.show()

## 3. Convergence Summary

In [ ]:
rows = []
for k in k_values:
    s = serial_df[serial_df['k'] == k]
    d = dist_df[dist_df['k'] == k]
    if len(s) == 0:
        continue
    rows.append({
        'k':               k,
        'serial_time_s':   round(float(s['time_seconds'].values[0]), 2),
        'dist_time_s':     round(float(d['time_seconds'].values[0]), 2) if len(d) > 0 else None,
        'speedup':         round(float(d['speedup'].values[0]), 3) if len(d) > 0 else None,
        'efficiency_%':    round(float(d['speedup'].values[0]) / n_workers * 100, 1) if len(d) > 0 else None,
        'serial_iters':    int(s['iterations'].values[0]),
        'dist_iters':      int(d['iterations'].values[0]) if len(d) > 0 else None,
        'serial_inertia':  round(float(s['inertia'].values[0]), 0),
        'dist_inertia':    round(float(d['inertia'].values[0]), 0) if len(d) > 0 else None,
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

In [ ]:
# Iterations to converge comparison
fig, ax = plt.subplots(figsize=(8, 4))
width = 0.35
x = np.arange(len(k_values))
ser_iters  = summary['serial_iters'].tolist()
dist_iters = summary['dist_iters'].tolist()
ax.bar(x - width / 2, ser_iters,  width, label='Serial',      color=COLORS[0], edgecolor='black')
ax.bar(x + width / 2, dist_iters, width, label='Distributed', color=COLORS[1], edgecolor='black')
ax.set_xticks(x)
ax.set_xticklabels([f'k={k}' for k in k_values])
ax.set_ylabel('Iterations to Converge')
ax.set_title('K-Means Convergence \u2014 Serial vs Distributed')
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('kmeans_convergence.png', bbox_inches='tight')
plt.show()

## 4. Interpretation

### MPI Performance

The distributed K-Means uses `MPI.Allreduce` to aggregate per-cluster sums and counts across all 4 processes each iteration — no separate broadcast step needed since Allreduce delivers results to all ranks simultaneously. Communication cost is **O(k × features) per iteration**, independent of dataset size (581k samples).

Speedup scales well because:
- **Assignment step** (dominant cost) is perfectly parallel — each process handles its ~145k sample partition independently.
- **Update step** Allreduce payload is small: k × 54 floats per call.

Differences in speedup across k values arise from the Allreduce payload growing proportionally with k.

### Inertia and Convergence

Both serial and distributed produce nearly identical inertia values for the same k — the algorithm is deterministic given the same centroid initialization seed. Minor differences can arise from floating-point accumulation order in the Allreduce reduction.

Inertia decreases as k increases — more clusters always fit the data tighter. The Covertype dataset contains **7 forest cover types**, so k=7 is the semantically natural choice and the point where the inertia reduction rate is expected to plateau (elbow).